## User-namespace remapping & rootless Docker

Running as non-root inside the container helps, but two deeper moves change *who the container's root maps to on the host*. Both shrink what a breakout gets you.

**User-namespace remapping.** By default, container UID 0 *is* host UID 0 — they share the user namespace. Remapping breaks that: the daemon maps container root to, say, host UID 100000, so from the host the container's "root" is a normal unprivileged user. A kernel-bug breakout still lands the attacker as **unprivileged**. Enable it in `/etc/docker/daemon.json`:

```json
{ "userns-remap": "default" }
```

`"default"` creates a `dockremap` user with a `/etc/subuid`/`subgid` range and remaps all containers into it. Trade-offs: **all** containers share one remap range (no per-container isolation), **bind mounts get fiddly** (host files owned by `100000` look like root inside), and `--privileged` and host IPC/PID sharing become incompatible. Leave it off for dev/CI; turn it on for servers running less-trusted images.

**Rootless Docker** goes further: `userns-remap` still needs the **daemon** to run as root, but rootless runs the *entire daemon and all containers* under your unprivileged account.

```bash
dockerd-rootless-setuptool.sh install   # one-time
systemctl --user start docker
```

Now there's **no root daemon to compromise** — a container escape only gets an attacker *your* UID's privileges. Trade-offs: networking is slower (userspace `slirp4netns`), `--network host` and `--privileged` are unavailable, cgroup v2 is required for limits, and there are no host `iptables` rules (publishing goes through `slirp4netns`). Rootless is the right answer for **shared dev hosts, sandboxes, and CI runners** — exactly where a compromised step shouldn't mean a compromised host.